# Atividade 4 - Tolerancia a Falhas e Monitoramento

## API de gerenciamento de mercado no Kubernetes

Este notebook executa e registra os experimentos. As saidas devem ser geradas no ambiente e usadas como evidencias reais no relatorio. 


## 1. Ambiente

Registre: sistema operacional, versoes de Docker/kubectl/Minikube/Helm, driver, CPU e memoria alocadas ao cluster. Para instalar Minikube e Helm no Windows, use os instaladores oficiais ou `winget install Kubernetes.minikube` e `winget install Helm.Helm`.

O kernel selecionado no VS Code precisa do pacote `ipykernel`. Se a primeira celula de codigo indicar que ele esta ausente, instale-o no ambiente Python selecionado com:

```powershell
python -m pip install ipykernel
```

Com Minikube instalado, inicie o cluster e habilite a fonte de metricas requerida pelo HPA:

```powershell
minikube start --driver=docker --cpus=4 --memory=6144
minikube addons enable metrics-server
kubectl config use-context minikube
```

A stack kube-prometheus-stack sera instalada com Helm. O Prometheus standalone que possa estar ouvindo em `localhost:9090` fica intacto; este notebook publica o Prometheus do cluster em `localhost:19091`.

In [1]:
import json
import pathlib
import shutil
import socket
import subprocess
import sys
import time

ROOT = pathlib.Path.cwd()
if not (ROOT / 'k8s' / '01-databases.yaml').exists():
    raise RuntimeError('Abra este notebook com a raiz do repositorio como diretorio de trabalho.')

def run(args, check=True):
    result = subprocess.run([str(arg) for arg in args], cwd=ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    if check and result.returncode:
        raise RuntimeError(f'Comando retornou {result.returncode}: {args}')
    return result.stdout.strip()

context = run(['kubectl', 'config', 'current-context'], check=False)
print('Contexto Kubernetes:', context or '(nao configurado)')
print('Minikube:', shutil.which('minikube') or 'nao instalado')
print('Helm:', shutil.which('helm') or 'nao instalado')
run(['kubectl', 'get', 'nodes', '-o', 'wide'], check=False)

minikube
Contexto Kubernetes: minikube
Minikube: C:\Program Files\Kubernetes\Minikube\minikube.EXE
Helm: C:\Users\vinic\AppData\Local\Microsoft\WinGet\Packages\Helm.Helm_Microsoft.Winget.Source_8wekyb3d8bbwe\windows-amd64\helm.EXE
NAME       STATUS   ROLES           AGE   VERSION   INTERNAL-IP    EXTERNAL-IP   OS-IMAGE                         KERNEL-VERSION                             CONTAINER-RUNTIME
minikube   Ready    control-plane   45h   v1.37.0   192.168.49.2   <none>        Debian GNU/Linux 12 (bookworm)   6.6.87.2-microsoft-standard-WSL2 (amd64)   containerd://2.3.4


'NAME       STATUS   ROLES           AGE   VERSION   INTERNAL-IP    EXTERNAL-IP   OS-IMAGE                         KERNEL-VERSION                             CONTAINER-RUNTIME\nminikube   Ready    control-plane   45h   v1.37.0   192.168.49.2   <none>        Debian GNU/Linux 12 (bookworm)   6.6.87.2-microsoft-standard-WSL2 (amd64)   containerd://2.3.4'

## 2. Preparacao do cluster e do monitoramento

A atividade recomenda Minikube. Instale Minikube e Helm no Windows e, em um terminal, execute:

```powershell
minikube start --driver=docker --cpus=4 --memory=6144
minikube addons enable metrics-server
kubectl config use-context minikube
```

Confira o contexto nesta celula antes de seguir. A stack `kube-prometheus-stack` instala Prometheus Operator, Grafana, kube-state-metrics e node-exporter. O metrics-server e distinto do Prometheus e fornece a CPU usada pelo HPA. O Prometheus standalone em `localhost:9090` nao sera parado; o Prometheus do cluster usara `localhost:19091`.

Os manifests tambem funcionam no Kubernetes do Docker Desktop, mas registre esse ambiente corretamente no relatorio se nao executar Minikube.

In [2]:
run(['helm', 'repo', 'add', 'prometheus-community', 'https://prometheus-community.github.io/helm-charts'])
run(['helm', 'repo', 'update'])
run(['helm', 'upgrade', '--install', 'monitoring-stack', 'prometheus-community/kube-prometheus-stack', '--namespace', 'monitoring', '--create-namespace', '--wait', '--timeout', '10m'])
run(['kubectl', 'get', 'pods', '-n', 'monitoring'])

"prometheus-community" already exists with the same configuration, skipping
Hang tight while we grab the latest from your chart repositories...
...Successfully got an update from the "prometheus-community" chart repository
Update Complete. âŽˆHappy Helming!âŽˆ
Release "monitoring-stack" has been upgraded. Happy Helming!
NAME: monitoring-stack
LAST DEPLOYED: Thu Oct  1 12:13:50 2026
NAMESPACE: monitoring
STATUS: deployed
REVISION: 2
DESCRIPTION: Upgrade complete
NOTES:
kube-prometheus-stack has been installed. Check its status by running:
  kubectl --namespace monitoring get pods -l "release=monitoring-stack"

Get Grafana 'admin' user password by running:

  kubectl --namespace monitoring get secrets monitoring-stack-grafana -o jsonpath="{.data.admin-password}" | base64 -d ; echo

Access Grafana local instance:

  export POD_NAME=$(kubectl --namespace monitoring get pod -l "app.kubernetes.io/name=grafana,app.kubernetes.io/instance=monitoring-stack" -oname)
  kubectl --namespace monitori

'NAME                                                     READY   STATUS    RESTARTS       AGE\nalertmanager-monitoring-stack-kube-prom-alertmanager-0   2/2     Running   2 (127m ago)   45h\nmonitoring-stack-grafana-5848cff544-g4b2d                3/3     Running   0              31s\nmonitoring-stack-kube-prom-operator-5b8c74fff8-2jxgg     1/1     Running   1 (127m ago)   45h\nmonitoring-stack-kube-state-metrics-d8ccff6f8-qj895      1/1     Running   1 (127m ago)   45h\nmonitoring-stack-prometheus-node-exporter-6thtr          1/1     Running   1 (127m ago)   45h\nprometheus-monitoring-stack-kube-prom-prometheus-0       2/2     Running   2 (127m ago)   44h'

In [13]:
images = {
    'api-mercado/ms-funcionarios:atividade4': './src/services/ms-funcionarios',
    'api-mercado/ms-produtos:atividade4': './src/services/ms-produtos',
    'api-mercado/ms-vendas:atividade4': './src/services/ms-vendas',
    'api-mercado/frontend:atividade4': './frontend',
}
for image, build_context in images.items():
    run(['docker', 'build', '-t', image, build_context])
    if context == 'minikube':
        run(['minikube', 'image', 'load', image])

# Os bancos precisam estar prontos antes de as APIs importarem os modelos.
run(['kubectl', 'apply', '-f', 'k8s/01-databases.yaml'])
for name in ['db-funcionarios', 'db-produtos', 'db-vendas']:
    run(['kubectl', 'rollout', 'status', f'statefulset/{name}', '-n', 'mercado', '--timeout=180s'])

run(['kubectl', 'apply', '-f', 'k8s/02-applications.yaml'])
run(['kubectl', 'apply', '-f', 'k8s/03-postgres-exporters.yaml'])
for name in ['ms-funcionarios', 'ms-produtos', 'ms-vendas', 'frontend', 'nginx', 'nginx-exporter', 'postgres-exporter-funcionarios', 'postgres-exporter-produtos', 'postgres-exporter-vendas']:
    run(['kubectl', 'rollout', 'status', f'deployment/{name}', '-n', 'mercado', '--timeout=180s'])
run(['kubectl', 'get', 'pods', '-n', 'mercado', '-o', 'wide'])
run(['kubectl', 'get', 'hpa', '-n', 'mercado'])
run(['kubectl', 'top', 'pods', '-n', 'mercado'], check=False)

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 1.70kB 0.0s done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 1.9s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [builder 1/5] FROM docker.io/library/python:3.11-slim@sha256:e41613d42d4891e4930f79523f93f81bbc7632584ec65e36ab055f41a800b41e
#4 resolve docker.io/library/python:3.11-slim@sha256:e41613d42d4891e4930f79523f93f81bbc7632584ec65e36ab055f41a800b41e 0.0s done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 1.46kB 0.0s done
#5 DONE 0.0s

#6 [builder 3/5] RUN apt-get update && apt-get install -y --no-install-recommends     gcc     libpq-dev     && rm -rf /var/lib/apt/lists/*
#6 CACHED

#7 [builder 4/5] COPY requirements.txt .
#7 CACHED

#8 [builder 2/5] WORKDIR /app
#8 CACHED

#9 [runner 3/7] RUN apt-get update && apt-get install -y --no-

namespace/mercado created
secret/database-credentials created
service/db-funcionarios created
statefulset.apps/db-funcionarios created
service/db-produtos created
statefulset.apps/db-produtos created
service/db-vendas created
statefulset.apps/db-vendas created
Waiting for 1 pods to be ready...
partitioned roll out complete: 1 new pods have been updated...
partitioned roll out complete: 1 new pods have been updated...
partitioned roll out complete: 1 new pods have been updated...
deployment.apps/ms-funcionarios created
service/ms-funcionarios created
servicemonitor.monitoring.coreos.com/ms-funcionarios created
deployment.apps/ms-produtos created
service/ms-produtos created
servicemonitor.monitoring.coreos.com/ms-produtos created
deployment.apps/ms-vendas created
service/ms-vendas created
servicemonitor.monitoring.coreos.com/ms-vendas created
deployment.apps/frontend created
service/frontend created
configmap/nginx-config created
deployment.apps/nginx created
service/nginx created
deploy

'NAME                CPU(cores)   MEMORY(bytes)   \ndb-funcionarios-0   115m         24Mi            \ndb-produtos-0       123m         26Mi            \ndb-vendas-0         129m         28Mi'

## 3. Experimentos 1 e 2

### Delecao manual de Pod

A celula seguinte apaga um pod de `ms-produtos` e mede o tempo ate um pod com nova identidade estar Running/Ready e duas replicas estarem prontas. Registre tambem os eventos e a consulta de pods no Prometheus.

### Falha no processo do container

A segunda celula encerra o PID 1 de um container. Espera-se o mesmo UID de Pod e o contador `RESTARTS` aumentado. Isso difere da delecao manual: nesse caso a Deployment cria outro Pod, com identidade nova.

In [3]:
def product_pods():
    data = json.loads(run(['kubectl', 'get', 'pods', '-n', 'mercado', '-l', 'app=ms-produtos', '-o', 'json']))
    return data['items']

before = product_pods()
old_names = {pod['metadata']['name'] for pod in before}
target = before[0]['metadata']['name']
print('Pods antes:', sorted(old_names))
started = time.perf_counter()
run(['kubectl', 'delete', 'pod', target, '-n', 'mercado'])
replacement = None
while time.perf_counter() - started < 180:
    ready = [pod for pod in product_pods() if pod['status'].get('phase') == 'Running' and all(c.get('ready') for c in pod['status'].get('containerStatuses', []))]
    candidates = [pod for pod in ready if pod['metadata']['name'] not in old_names]
    if candidates and len(ready) >= 2:
        replacement = candidates[0]
        break
    time.sleep(2)
print(f'Tempo de recuperacao: {time.perf_counter() - started:.1f} s')
print('Pod substituto:', replacement['metadata']['name'] if replacement else 'nao observado em 180 s')
run(['kubectl', 'get', 'pods', '-n', 'mercado', '-l', 'app=ms-produtos', '-o', 'wide'])
run(['kubectl', 'get', 'events', '-n', 'mercado', '--sort-by=.lastTimestamp'])

{
    "apiVersion": "v1",
    "items": [
        {
            "apiVersion": "v1",
            "kind": "Pod",
            "metadata": {
                "creationTimestamp": "2026-09-29T18:24:14Z",
                "generateName": "ms-produtos-548488d5b7-",
                "generation": 1,
                "labels": {
                    "app": "ms-produtos",
                    "pod-template-hash": "548488d5b7"
                },
                "name": "ms-produtos-548488d5b7-jlbqd",
                "namespace": "mercado",
                "ownerReferences": [
                    {
                        "apiVersion": "apps/v1",
                        "blockOwnerDeletion": true,
                        "controller": true,
                        "kind": "ReplicaSet",
                        "name": "ms-produtos-548488d5b7",
                        "uid": "cfe227c8-1654-449b-b51e-c403adf9bdb7"
                    }
                ],
                "resourceVersion": "5959",
          

'LAST SEEN   TYPE     REASON             OBJECT                                 MESSAGE\n20m         Normal   SuccessfulCreate   replicaset/ms-funcionarios-75bd5674d   Created pod: ms-funcionarios-75bd5674d-8jlf4\n20m         Normal   Killing            pod/ms-funcionarios-75bd5674d-b47vj    Stopping container api\n20m         Normal   Scheduled          pod/ms-funcionarios-75bd5674d-8jlf4    Successfully assigned mercado/ms-funcionarios-75bd5674d-8jlf4 to minikube\n20m         Normal   Created            pod/ms-funcionarios-75bd5674d-8jlf4    Container created\n20m         Normal   Started            pod/ms-funcionarios-75bd5674d-8jlf4    Container started\n20m         Normal   Pulled             pod/ms-funcionarios-75bd5674d-8jlf4    Container image "api-mercado/ms-funcionarios:atividade4" already present on machine and can be accessed by the pod\n15m         Normal   Killing            pod/ms-funcionarios-75bd5674d-8jlf4    Stopping container api\n15m         Normal   Scheduled     

In [4]:
pod = product_pods()[0]
pod_name = pod['metadata']['name']
old_uid = pod['metadata']['uid']
container = pod['status']['containerStatuses'][0]
old_restarts = container['restartCount']
print(f'Falha controlada em {pod_name}; UID={old_uid}; RESTARTS={old_restarts}')
started = time.perf_counter()
run(['kubectl', 'exec', '-n', 'mercado', pod_name, '-c', container['name'], '--', 'sh', '-c', 'kill 1'], check=False)
restarted = False
while time.perf_counter() - started < 120:
    current = {item['metadata']['name']: item for item in product_pods()}.get(pod_name)
    if current:
        statuses = current['status'].get('containerStatuses', [])
        if current['status'].get('phase') == 'Running' and statuses and statuses[0].get('restartCount', 0) > old_restarts:
            restarted = True
            break
    time.sleep(2)
print(f'Tempo ate reinicio: {time.perf_counter() - started:.1f} s; reinicio observado={restarted}')
current = next((item for item in product_pods() if item['metadata']['name'] == pod_name), None)
if current:
    print('Mesmo UID:', current['metadata']['uid'] == old_uid, '| RESTARTS:', current['status']['containerStatuses'][0]['restartCount'])
run(['kubectl', 'get', 'pods', '-n', 'mercado', '-l', 'app=ms-produtos', '-o', 'wide'])

{
    "apiVersion": "v1",
    "items": [
        {
            "apiVersion": "v1",
            "kind": "Pod",
            "metadata": {
                "creationTimestamp": "2026-10-01T15:15:45Z",
                "generateName": "ms-produtos-548488d5b7-",
                "generation": 1,
                "labels": {
                    "app": "ms-produtos",
                    "pod-template-hash": "548488d5b7"
                },
                "name": "ms-produtos-548488d5b7-v72jb",
                "namespace": "mercado",
                "ownerReferences": [
                    {
                        "apiVersion": "apps/v1",
                        "blockOwnerDeletion": true,
                        "controller": true,
                        "kind": "ReplicaSet",
                        "name": "ms-produtos-548488d5b7",
                        "uid": "cfe227c8-1654-449b-b51e-c403adf9bdb7"
                    }
                ],
                "resourceVersion": "13364",
         

'NAME                           READY   STATUS    RESTARTS       AGE   IP            NODE       NOMINATED NODE   READINESS GATES\nms-produtos-548488d5b7-v72jb   0/1     Running   1 (2s ago)     41s   10.244.0.33   minikube   <none>           <none>\nms-produtos-548488d5b7-zqzz4   1/1     Running   2 (129m ago)   44h   10.244.0.21   minikube   <none>           <none>'

## 4. Experimento 3 - Sobrecarga de CPU e HPA

A carga e gerada por processos Python dentro dos containers da API `ms-produtos`, para que o consumo seja medido no pod que o HPA acompanha. O HPA esta configurado com minimo de 2, maximo de 5 replicas e CPU alvo de 60%. A celula encerra os processos ao final, mesmo se a espera por escalonamento expirar. Confirme que `kubectl top pods` fornece metricas antes do teste.

In [5]:
pods = product_pods()
load_pods = [pod['metadata']['name'] for pod in pods if pod['status'].get('phase') == 'Running']
initial_replicas = len(load_pods)
print('Replicas iniciais:', initial_replicas, load_pods)
launcher = "import subprocess,sys; p=subprocess.Popen([sys.executable,'-c','while True: pass'],start_new_session=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); open('/tmp/atividade4-cpu.pid','w').write(str(p.pid))"
started = time.perf_counter()
started_on = []
try:
    for name in load_pods:
        run(['kubectl', 'exec', '-n', 'mercado', name, '-c', 'api', '--', 'python', '-c', launcher])
        started_on.append(name)
    scaled_at = None
    max_ready = initial_replicas
    deadline = time.perf_counter() + 180
    while time.perf_counter() < deadline:
        ready_text = subprocess.run(['kubectl', 'get', 'deployment', 'ms-produtos', '-n', 'mercado', '-o', 'jsonpath={.status.readyReplicas}'], cwd=ROOT, text=True, capture_output=True).stdout.strip()
        ready = int(ready_text or '0')
        max_ready = max(max_ready, ready)
        if ready > initial_replicas and scaled_at is None:
            scaled_at = time.perf_counter()
        if ready >= 5:
            break
        time.sleep(5)
    print('Tempo ate primeira replica adicional:', f'{scaled_at - started:.1f} s' if scaled_at else 'nao escalonou no limite de 180 s')
    print('Maior quantidade Ready observada:', max_ready, '| maximo configurado: 5')
    run(['kubectl', 'get', 'hpa', 'ms-produtos', '-n', 'mercado', '-o', 'wide'])
    run(['kubectl', 'top', 'pods', '-n', 'mercado', '-l', 'app=ms-produtos'], check=False)
finally:
    for name in started_on:
        run(['kubectl', 'exec', '-n', 'mercado', name, '-c', 'api', '--', 'python', '-c', "import os,signal; os.kill(int(open('/tmp/atividade4-cpu.pid').read()), signal.SIGTERM)"], check=False)

print('Carga removida; aguardando o HPA voltar ao minimo.')
scale_down_started = time.perf_counter()
ready = max_ready
while time.perf_counter() - scale_down_started < 240:
    ready_text = subprocess.run(['kubectl', 'get', 'deployment', 'ms-produtos', '-n', 'mercado', '-o', 'jsonpath={.status.readyReplicas}'], cwd=ROOT, text=True, capture_output=True).stdout.strip()
    ready = int(ready_text or '0')
    if ready <= initial_replicas:
        break
    time.sleep(5)
print(f'Tempo para voltar a {initial_replicas} replicas: {time.perf_counter() - scale_down_started:.1f} s; prontas={ready}')
run(['kubectl', 'get', 'hpa', 'ms-produtos', '-n', 'mercado', '-o', 'wide'])
run(['kubectl', 'get', 'events', '-n', 'mercado', '--sort-by=.lastTimestamp'])

{
    "apiVersion": "v1",
    "items": [
        {
            "apiVersion": "v1",
            "kind": "Pod",
            "metadata": {
                "creationTimestamp": "2026-10-01T15:15:45Z",
                "generateName": "ms-produtos-548488d5b7-",
                "generation": 1,
                "labels": {
                    "app": "ms-produtos",
                    "pod-template-hash": "548488d5b7"
                },
                "name": "ms-produtos-548488d5b7-v72jb",
                "namespace": "mercado",
                "ownerReferences": [
                    {
                        "apiVersion": "apps/v1",
                        "blockOwnerDeletion": true,
                        "controller": true,
                        "kind": "ReplicaSet",
                        "name": "ms-produtos-548488d5b7",
                        "uid": "cfe227c8-1654-449b-b51e-c403adf9bdb7"
                    }
                ],
                "resourceVersion": "13411",
         

'LAST SEEN   TYPE      REASON              OBJECT                                 MESSAGE\n27m         Normal    Scheduled           pod/ms-funcionarios-75bd5674d-8jlf4    Successfully assigned mercado/ms-funcionarios-75bd5674d-8jlf4 to minikube\n27m         Normal    SuccessfulCreate    replicaset/ms-funcionarios-75bd5674d   Created pod: ms-funcionarios-75bd5674d-8jlf4\n27m         Normal    Killing             pod/ms-funcionarios-75bd5674d-b47vj    Stopping container api\n27m         Normal    Created             pod/ms-funcionarios-75bd5674d-8jlf4    Container created\n27m         Normal    Pulled              pod/ms-funcionarios-75bd5674d-8jlf4    Container image "api-mercado/ms-funcionarios:atividade4" already present on machine and can be accessed by the pod\n27m         Normal    Started             pod/ms-funcionarios-75bd5674d-8jlf4    Container started\n22m         Normal    Killing             pod/ms-funcionarios-75bd5674d-8jlf4    Stopping container api\n22m         Normal 

## 5. Experimento 4 - Interrupcao do Prometheus

Antes dos experimentos, libere as portas locais se o Compose estiver ativo (`docker compose stop`, sem remover volumes). Em terminais separados, inicie os acessos:

```powershell
kubectl port-forward -n mercado svc/api-gateway 8080:80
kubectl port-forward -n mercado svc/frontend 5173:5173
kubectl port-forward -n monitoring svc/monitoring-stack-kube-prometheus-prometheus 19091:9090
kubectl port-forward -n monitoring svc/monitoring-stack-grafana 13000:80
```

Abra `http://localhost:5173`, API `http://localhost:8080`, Prometheus `http://localhost:19091` e Grafana `http://localhost:13000` (credenciais iniciais do chart: `admin` / `prom-operator`). Consultas PromQL para as evidencias:

- Pods ativos: `sum(kube_pod_status_phase{namespace="mercado",phase="Running"})`
- CPU por pod: `sum by (pod) (rate(container_cpu_usage_seconds_total{namespace="mercado",pod=~"ms-produtos-.*",container!="POD",container!=""}[2m]))`
- Reinicios: `kube_pod_container_status_restarts_total{namespace="mercado"}`
- HPA atual/desejado: `kube_horizontalpodautoscaler_status_current_replicas{namespace="mercado"}` e `kube_horizontalpodautoscaler_status_desired_replicas{namespace="mercado"}`
- Requisicoes das APIs: `rate(http_requests_total[2m])`; bancos: `pg_up`; Nginx: `nginx_up`

A celula abaixo remove o pod Prometheus e mede sua volta. Durante a lacuna, ela consulta `/healthz` diretamente dentro de um pod da API para evidenciar que monitoramento e aplicacao sao componentes independentes.

In [15]:
prometheus_pods = json.loads(run(['kubectl', 'get', 'pods', '-n', 'monitoring', '-l', 'app.kubernetes.io/name=prometheus', '-o', 'json']))['items']
if not prometheus_pods:
    raise RuntimeError('Pod Prometheus nao encontrado; confira a instalacao Helm e os labels no namespace monitoring.')
old_name = prometheus_pods[0]['metadata']['name']
print('Prometheus antes da falha:', old_name)
started = time.perf_counter()
run(['kubectl', 'delete', 'pod', old_name, '-n', 'monitoring'])
api_pod = product_pods()[0]['metadata']['name']
print('Aplicacao durante a interrupcao do monitoramento:')
run(['kubectl', 'exec', '-n', 'mercado', api_pod, '-c', 'api', '--', 'python', '-c', "import urllib.request; print('healthz HTTP', urllib.request.urlopen('http://localhost:8001/healthz', timeout=5).status)"])
replacement = None
while time.perf_counter() - started < 240:
    candidates = json.loads(run(['kubectl', 'get', 'pods', '-n', 'monitoring', '-l', 'app.kubernetes.io/name=prometheus', '-o', 'json']))['items']
    ready = [pod for pod in candidates if pod['status'].get('phase') == 'Running' and all(c.get('ready') for c in pod['status'].get('containerStatuses', []))]
    if ready:
        replacement = ready[0]
        break
    time.sleep(3)
print(f'Tempo para Prometheus voltar Ready: {time.perf_counter() - started:.1f} s')
print('Pod Prometheus Ready:', replacement['metadata']['name'] if replacement else 'nao observado em 240 s')
run(['kubectl', 'get', 'pods', '-n', 'mercado', '-o', 'wide'])
run(['kubectl', 'get', 'pods', '-n', 'monitoring', '-l', 'app.kubernetes.io/name=prometheus', '-o', 'wide'])
run(['kubectl', 'get', 'events', '-n', 'monitoring', '--sort-by=.lastTimestamp'])

{
    "apiVersion": "v1",
    "items": [
        {
            "apiVersion": "v1",
            "kind": "Pod",
            "metadata": {
                "annotations": {
                    "kubectl.kubernetes.io/default-container": "prometheus"
                },
                "creationTimestamp": "2026-09-29T18:04:39Z",
                "generateName": "prometheus-monitoring-stack-kube-prom-prometheus-",
                "generation": 1,
                "labels": {
                    "app.kubernetes.io/instance": "monitoring-stack-kube-prom-prometheus",
                    "app.kubernetes.io/managed-by": "prometheus-operator",
                    "app.kubernetes.io/name": "prometheus",
                    "app.kubernetes.io/version": "3.15.0-distroless",
                    "apps.kubernetes.io/pod-index": "0",
                    "controller-revision-hash": "prometheus-monitoring-stack-kube-prom-prometheus-74b85cbdcf",
                    "operator.prometheus.io/name": "monitoring-st

'LAST SEEN   TYPE      REASON              OBJECT                                                             MESSAGE\n38m         Normal    SuccessfulCreate    job/monitoring-stack-kube-prom-admission-create                    Created pod: monitoring-stack-kube-prom-admission-create-ddxrr\n38m         Normal    Scheduled           pod/monitoring-stack-kube-prom-admission-create-ddxrr              Successfully assigned monitoring/monitoring-stack-kube-prom-admission-create-ddxrr to minikube\n38m         Normal    Pulling             pod/monitoring-stack-kube-prom-admission-create-ddxrr              Pulling image "ghcr.io/jkroepke/kube-webhook-certgen:1.8.9"\n38m         Normal    Started             pod/monitoring-stack-kube-prom-admission-create-ddxrr              Container started\n38m         Normal    Created             pod/monitoring-stack-kube-prom-admission-create-ddxrr              Container created\n38m         Normal    Pulled              pod/monitoring-stack-kube-prom-admi

## 6. Relatorio individual (4 a 6 paginas)

Use as saidas deste notebook e capturas do Prometheus/Grafana. Cada imagem deve ter legenda com o experimento, o instante e a conclusao que evidencia. Exporte o notebook preenchido para PDF e revise a contagem de paginas.

| Experimento | Tempo medido | Evidencias sugeridas |
| --- | ---: | --- |
| Delecao do pod | Preencher com a saida | Pod removido e substituto Ready; eventos; pods ativos no Prometheus |
| Falha no processo | Preencher com a saida | Comando de falha; mesmo UID; `RESTARTS` aumentado |
| Sobrecarga/HPA | Preencher com a saida | CPU; replicas inicial/final; maximo observado; reducao apos retirar carga |
| Interrupcao Prometheus | Preencher com a saida | pod Prometheus substituido; API saudavel durante a falha; coleta retomada |

Inclua ambiente e versoes, configuracao de Deployment/requests/limits/probes/HPA, analise dos tempos medidos, limites do cluster local e a diferenca entre falha de monitoramento e falha da aplicacao. Confirme com o docente que este projeto atende ao requisito de aplicacao diferente das atividades anteriores.